In [ ]:
!pip install groq python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 10.7 MB/s eta 0:00:00


In [ ]:
import os
import getpass
from groq import Groq
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

Enter your Groq API key: ··········


In [ ]:
MODEL = "llama-3.3-70b-versatile"

MODEL_CONFIG = {
    "technical": {
        "system_prompt": """You are a Technical Support Expert.
Be precise, logical, and code-focused.
Diagnose errors and provide clear fixes with examples."""
    },
    "billing": {
        "system_prompt": """You are a Billing Support Expert.
Be empathetic, polite, and policy-driven.
Help with refunds, charges, and subscriptions."""
    },
    "general": {
        "system_prompt": """You are a helpful general assistant.
Answer casually and clearly."""
    }
}

In [ ]:
def route_prompt(user_input):
    prompt = f"""
Classify the following user query into one of these categories:
[technical, billing, general, tool]

Use 'tool' for:
- real-time data (prices, weather, stock, etc.)

Return ONLY one word.

Query: {user_input}
"""

    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip().lower()

In [ ]:
def process_request(user_input):
    category = route_prompt(user_input)

    print(f"[Router] → {category}")

    #  TOOL HANDLING
    if category == "tool":
        return get_bitcoin_price()

    # fallback safety
    if category not in MODEL_CONFIG:
        category = "general"

    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    response = client.chat.completions.create(
        model=MODEL,
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )

    return response.choices[0].message.content

In [ ]:
print(process_request("My python script is throwing an IndexError"))

[Router] → technical
**IndexError in Python**

An `IndexError` in Python occurs when you try to access an element in a sequence (such as a list, tuple, or string) using an index that is out of range.

**Common Causes**
-----------------

1. **Out of bounds indexing**: Trying to access an element at an index that is greater than or equal to the length of the sequence.
2. **Negative indexing**: Using a negative index that is less than `-length` of the sequence.
3. **Empty sequence**: Trying to access an element in an empty sequence.

**Example Error**
-----------------

```python
my_list = [1, 2, 3]
print(my_list[3])  # Raises IndexError: list index out of range
```

**Debugging Steps**
-------------------

1. **Check the sequence length**: Verify that the index you are using is within the bounds of the sequence.
2. **Verify the sequence is not empty**: Make sure the sequence is not empty before trying to access its elements.
3. **Use try-except blocks**: Use try-except blocks to catch a

In [ ]:
print(process_request("I was charged twice for my subscription"))

[Router] → billing
I'm so sorry to hear that you were charged twice for your subscription. I can imagine how frustrating that must be for you. I'm here to help you resolve this issue as quickly as possible.

To assist you, I'll need to look into your account and investigate the double charge. Can you please provide me with your subscription details, such as your account name, subscription plan, and the date of the double charge?

Additionally, I'll need to verify some information to ensure that I'm speaking with the account owner. Can you please confirm your email address associated with the subscription and the last 4 digits of the payment method used?

Once I have this information, I'll be happy to help you with a refund for the duplicate charge and ensure that your subscription is updated correctly. Your satisfaction is my top priority, and I appreciate your patience and cooperation in resolving this matter.


In [ ]:
def get_bitcoin_price():
    # mock function (no API needed)
    return "Bitcoin is currently $65,000 (mock value)"

In [ ]:
print(process_request("What is the current price of Bitcoin?"))

[Router] → tool
Bitcoin is currently $65,000 (mock value)


In [ ]:
print(process_request("Who won the FIFA World Cup 2022?"))

[Router] → general
The winner of the FIFA World Cup 2022 was Argentina. They defeated France 4-2 in a penalty shootout, after the game had ended 3-3 after extra time, in the final on December 18, 2022. It was a thrilling match, and Argentina's victory marked their third World Cup title. Lionel Messi was also named the tournament's best player.


In [ ]:
print(process_request("Explain black holes in simple terms"))

[Router] → general
Black holes are basically super-powerful vacuum cleaners in space. Here's how they work:

1. **What is a black hole?**: A black hole is a region in space where the gravitational pull is so strong that nothing, including light, can escape. It's formed when a massive star collapses in on itself.

2. **How strong is the gravity?**: Imagine you have a super-strong magnet that pulls everything towards it. Now, imagine that magnet is so strong that it warps the fabric of space and time around it. That's basically what a black hole does, but instead of a magnet, it's gravity that's doing the pulling.

3. **What happens if you get too close?**: If you get too close to a black hole, you'll get sucked in. Once you cross a certain point, called the event horizon, there's no escape. You'll be pulled towards the center of the black hole, where you'll be crushed by the intense gravity.

4. **Can we see black holes?**: Since not even light can escape from a black hole, it's invisib